In [30]:
#TODO: load data
#TODO: train val test split
#TODO: missing data
#TODO: transformations/skewness
#TODO: correlation
#TODO: scaling

In [31]:
#imports
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.stats import skew

### Load data

In [32]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Find MAL folder no matter if the notebook is run from MAL/ or MAL/notebooks/
current_dir = Path.cwd()
MAL_DIR = current_dir if (current_dir / "data").exists() else current_dir.parent

DATA_DIR = MAL_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

# Later, when the data team gives us a final file, we only need to change this path.
DATASET_CANDIDATES = [
    PROCESSED_DATA_DIR / "focus_dataset.csv",
    DATA_DIR / "mock" / "new" / "data.csv",
    RAW_DATA_DIR / "room_conditions.csv",
]

dataset_path = next((path for path in DATASET_CANDIDATES if path.exists()), None)

if dataset_path is None:
    raise FileNotFoundError("No dataset found. Check data/raw, data/interim, or data/processed.")

df = pd.read_csv(dataset_path)

# Small generic cleanup that is safe for most CSVs
df = df.drop(columns=[col for col in df.columns if col.lower().startswith("unnamed")], errors="ignore")
df.columns = df.columns.str.strip()
df = df.reset_index(drop=True)

print(f"Loaded dataset: {dataset_path}")
print(f"Shape: {df.shape}")

display(df.head())
display(df.dtypes)
display(df.isna().sum().rename("missing_values"))


Loaded dataset: c:\Users\piotr\Desktop\University\Semester4\SEP4\SEP4\MAL\data\processed\focus_dataset.csv
Shape: (1811, 22)


,timePeriod,currentTemperature,currentHumidity,currentCO2,currentLight,currentNoise,maxTemp,minTemp,meanTemp,maxHumidity,...,maxCO2,minCO2,meanCO2,maxLight,minLight,meanLight,maxNoise,minNoise,meanNoise,rating
0,2026-04-01 08:49:00 to 2026-04-01 09:19:00,22,58,815,325,56,23,22,22.05,58,...,848,631,723.55,325,306,316.00,62,43,51.20,4
1,2026-04-01 10:39:00 to 2026-04-01 11:09:00,26,47,751,799,60,26,26,26.00,48,...,765,699,728.80,801,782,790.10,69,53,59.70,3
2,2026-04-02 19:15:00 to 2026-04-02 19:45:00,24,61,1230,630,56,27,24,25.39,61,...,1230,702,906.32,637,617,627.35,56,36,47.32,2
3,2026-04-05 02:57:00 to 2026-04-05 03:27:00,19,52,825,597,35,20,19,19.25,53,...,932,691,794.36,614,595,605.57,54,34,43.75,3
4,2026-04-05 07:32:00 to 2026-04-05 08:02:00,24,54,1534,658,34,27,23,25.00,71,...,1534,782,1099.48,674,654,662.35,48,30,37.39,1


timePeriod                str
currentTemperature      int64
currentHumidity         int64
currentCO2              int64
currentLight            int64
currentNoise            int64
maxTemp                 int64
minTemp                 int64
meanTemp              float64
maxHumidity             int64
minHumidity             int64
meanHumidity          float64
maxCO2                  int64
minCO2                  int64
meanCO2               float64
maxLight                int64
minLight                int64
meanLight             float64
maxNoise                int64
minNoise                int64
meanNoise             float64
rating                  int64
dtype: object

timePeriod            0
currentTemperature    0
currentHumidity       0
currentCO2            0
currentLight          0
currentNoise          0
maxTemp               0
minTemp               0
meanTemp              0
maxHumidity           0
minHumidity           0
meanHumidity          0
maxCO2                0
minCO2                0
meanCO2               0
maxLight              0
minLight              0
meanLight             0
maxNoise              0
minNoise              0
meanNoise             0
rating                0
Name: missing_values, dtype: int64

### Train val test split

In [33]:
from sklearn.model_selection import train_test_split

TARGET_COLUMN = "rating"

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        "This dataset can be cleaned/preprocessed, but it cannot be split for supervised training yet."
    )

# Keep only numeric features for now.
# Later preprocessing steps can handle dates, categories, scaling, etc.
columns_to_ignore = [
    TARGET_COLUMN,
    "timePeriod",
    "session_id",
    "sent_at",
    "date",
    "Date",
    "id",
]

feature_columns = [
    column for column in df.columns
    if column not in columns_to_ignore and pd.api.types.is_numeric_dtype(df[column])
]

X = df[feature_columns].copy()
y = df[TARGET_COLUMN].copy()

# Remove rows where the target is missing.
valid_target_rows = y.notna()
X = X[valid_target_rows]
y = y[valid_target_rows]

# For now rating is treated as classes.
# If we later test regression, we can still reuse the same split and change the model/evaluation.
y = y.astype(int)

# Stratify only works if every class has at least 2 rows.
class_counts = y.value_counts()
stratify_target = y if class_counts.min() >= 2 else None

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=stratify_target,
)

train_val_stratify = y_train_val if y_train_val.value_counts().min() >= 2 else None

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765,  # gives roughly 15% validation, 70% train, 15% test overall
    random_state=42,
    stratify=train_val_stratify,
)

print("Feature columns:")
print(feature_columns)

print("\nSplit sizes:")
print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Validation: {X_val.shape}, {y_val.shape}")
print(f"Test: {X_test.shape}, {y_test.shape}")

print("\nTarget distribution:")
display(pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "validation": y_val.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}).fillna(0))

Feature columns:
['currentTemperature', 'currentHumidity', 'currentCO2', 'currentLight', 'currentNoise', 'maxTemp', 'minTemp', 'meanTemp', 'maxHumidity', 'minHumidity', 'meanHumidity', 'maxCO2', 'minCO2', 'meanCO2', 'maxLight', 'minLight', 'meanLight', 'maxNoise', 'minNoise', 'meanNoise']

Split sizes:
Train: (1267, 20), (1267,)
Validation: (272, 20), (272,)
Test: (272, 20), (272,)

Target distribution:


,train,validation,test
rating,,,
1,0.149171,0.150735,0.150735
2,0.198895,0.198529,0.198529
3,0.262036,0.261029,0.261029
4,0.217048,0.216912,0.216912
5,0.172849,0.172794,0.172794


### Missing data

In [34]:
#For missing values we decided to keep the first version simple: if a row has a missing value, we remove that row. We expect the final dataset to have very few or no missing values, because the sensor/data preparation step should produce complete rows. Dropping rows is easier to explain and avoids guessing values with mean/median imputation. If we later see that many rows are removed, then we should reconsider and use imputation instead.
# TODO: change this

missing_before = df.isna().sum().sum()
rows_before = len(df)

df = df.dropna().reset_index(drop=True)

rows_after = len(df)
rows_removed = rows_before - rows_after

print(f"Missing values before: {missing_before}")
print(f"Rows before: {rows_before}")
print(f"Rows removed: {rows_removed}")
print(f"Rows after: {rows_after}")

display(df.isna().sum().rename("missing_values_after"))


Missing values before: 0
Rows before: 1811
Rows removed: 0
Rows after: 1811


timePeriod            0
currentTemperature    0
currentHumidity       0
currentCO2            0
currentLight          0
currentNoise          0
maxTemp               0
minTemp               0
meanTemp              0
maxHumidity           0
minHumidity           0
meanHumidity          0
maxCO2                0
minCO2                0
meanCO2               0
maxLight              0
minLight              0
meanLight             0
maxNoise              0
minNoise              0
meanNoise             0
rating                0
Name: missing_values_after, dtype: int64

### Outliers

In [35]:
# tranformations/skewness
def handle_skewness(df, threshold=0.75):
    df_transformed = df.copy()
    
    # Select only numerical columns
    numeric_cols = df_transformed.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        skew_val = skew(df_transformed[col])
        
        # Highly Positive Skew (Right Tailed)
        if skew_val > threshold:
            # np.log1p handles log(0) by calculating log(1+x)
            df_transformed[col] = np.log1p(df_transformed[col])
            print(f"Applied Log Transform to '{col}' (Skew: {skew_val:.2f})")
            
        # Highly Negative Skew (Left Tailed)
        elif skew_val < -threshold:
            # Squaring often helps pull the left tail in
            df_transformed[col] = np.square(df_transformed[col])
            print(f"Applied Square Transform to '{col}' (Skew: {skew_val:.2f})")
            
    return df_transformed

# Apply to your features
X_train_val_skewed = handle_skewness(X_train_val)

Applied Log Transform to 'currentCO2' (Skew: 1.12)
Applied Log Transform to 'maxCO2' (Skew: 0.81)


In [36]:
#correlation


import pandas as pd
import numpy as np

def drop_highly_correlated(df, threshold=0.8):
    iteration = 1
    df_reduced = df.copy()
    
    while True:
        # 1. Calculate absolute correlation matrix
        corr_matrix = df_reduced.corr().abs()
        
        # 2. Select the upper triangle
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        # 3. Find pairs that exceed the threshold, sorted by highest correlation
        pairs = upper[upper > threshold].stack().dropna().sort_values(ascending=False)
        
        if pairs.empty:
            print("Optimization complete: No more features exceed the correlation threshold.")
            break
            
        # 4. Take the highest correlation pair
        (feat1, feat2), correlation_value = pairs.index[0], pairs.values[0]
        
        # 5. Drop the second feature in the pair
        # (Dropping one keeps the signal of the other without the redundancy)
        print(f"Iteration {iteration}: Dropping '{feat2}' (High correlation with '{feat1}': {correlation_value:.4f})")
        
        df_reduced = df_reduced.drop(columns=[feat2])
        iteration += 1
        
    return df_reduced

# Apply the logic to your skewed data
X_train_val_processed = drop_highly_correlated(X_train_val_skewed)

# Check the results
print(f"Original feature count: {X_train_val_skewed.shape[1]}")
print(f"Reduced feature count: {X_train_val_processed.shape[1]}")
X_train_val_processed.head()


Iteration 1: Dropping 'meanLight' (High correlation with 'currentLight': 0.9964)
Iteration 2: Dropping 'minLight' (High correlation with 'currentLight': 0.9944)
Iteration 3: Dropping 'maxLight' (High correlation with 'currentLight': 0.9918)
Iteration 4: Dropping 'meanNoise' (High correlation with 'maxNoise': 0.9677)
Iteration 5: Dropping 'meanTemp' (High correlation with 'currentTemperature': 0.9578)
Iteration 6: Dropping 'meanCO2' (High correlation with 'maxCO2': 0.9268)
Iteration 7: Dropping 'meanHumidity' (High correlation with 'currentHumidity': 0.9264)
Iteration 8: Dropping 'maxTemp' (High correlation with 'currentTemperature': 0.9211)
Iteration 9: Dropping 'minTemp' (High correlation with 'currentTemperature': 0.9189)
Iteration 10: Dropping 'minNoise' (High correlation with 'maxNoise': 0.8955)
Iteration 11: Dropping 'maxHumidity' (High correlation with 'currentHumidity': 0.8629)
Iteration 12: Dropping 'minHumidity' (High correlation with 'currentHumidity': 0.8552)
Iteration 13: D

,currentTemperature,currentHumidity,currentCO2,currentLight,currentNoise,minCO2
296,23,53,6.282267,365,37,486
1494,25,43,6.555357,225,76,606
1715,25,50,6.242223,598,52,504
70,23,46,6.333280,635,51,462
1740,20,40,6.922644,460,55,608


In [37]:
#scaling


scaler = StandardScaler()

# 2. Fit and transform the training data
X_train_val_scaled = scaler.fit_transform(X_train_val_processed)

# 3. Convert back to DataFrame to keep column names
X_train_val_final = pd.DataFrame(X_train_val_scaled, columns=X_train_val_processed.columns)

In [38]:
X_train_val_final

,currentTemperature,currentHumidity,currentCO2,currentLight,currentNoise,minCO2
0,0.136264,0.538663,-1.371102,-0.780382,-1.239824,-1.113602
1,0.868238,-0.704592,-0.423391,-1.593334,1.982881,-0.151985
2,0.868238,0.165687,-1.510066,0.572601,-0.000322,-0.969359
3,0.136264,-0.331615,-1.194071,0.787453,-0.082956,-1.305925
4,-0.961697,-1.077568,0.851213,-0.228736,0.247578,-0.135958
...,...,...,...,...,...,...
1534,-1.327684,0.911639,-0.990580,0.113865,1.239180,-1.458181
1535,-1.327684,1.781918,-0.350123,-1.169438,-0.248222,0.184581
1536,1.600212,1.781918,2.786905,-1.006847,0.908646,2.356231
1537,-0.595710,0.538663,0.186210,-1.111370,-0.578756,1.042022
